# 01 - Data Validation

Before building anything on top of this data, I want to be sure it's actually trustworthy —
no orphaned foreign keys, no unexpected nulls, and the train/test split structured. Loading and checking logic lives in `src/data_processing.py`;


In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import json
from data_processing import load_raw_data, validate_data, PROCESSED_DIR

In [7]:
data = load_raw_data()

for name, df in data.items():
    print(f"{name:12s} {df.shape}")

orders       (3421083, 7)
prior        (32434489, 4)
train        (1384617, 4)
products     (49688, 4)
aisles       (134, 2)
departments  (21, 2)


In [8]:
report = validate_data(data)
print(json.dumps(report, indent=2, default=str))

{
  "shapes": {
    "orders": [
      3421083,
      7
    ],
    "prior": [
      32434489,
      4
    ],
    "train": [
      1384617,
      4
    ],
    "products": [
      49688,
      4
    ],
    "aisles": [
      134,
      2
    ],
    "departments": [
      21,
      2
    ]
  },
  "nulls": {
    "orders": {
      "order_id": 0,
      "user_id": 0,
      "eval_set": 0,
      "order_number": 0,
      "order_dow": 0,
      "order_hour_of_day": 0,
      "days_since_prior_order": 206209
    },
    "prior": {
      "order_id": 0,
      "product_id": 0,
      "add_to_cart_order": 0,
      "reordered": 0
    },
    "train": {
      "order_id": 0,
      "product_id": 0,
      "add_to_cart_order": 0,
      "reordered": 0
    },
    "products": {
      "product_id": 0,
      "product_name": 0,
      "aisle_id": 0,
      "department_id": 0
    }
  },
  "eval_set_counts": {
    "prior": 3214874,
    "train": 131209,
    "test": 75000
  },
  "n_unique_users": 206209,
  "order_number_range

### Results

Ran this against the full Instacart dataset (206,209 users, 3.4M orders, 49,688 products).
Everything checks out clean:

- Zero orphaned order_ids or product_ids in either the prior or train files
- Aisle/department coverage is complete
- `days_since_prior_order` nulls line up exactly with each user's first order — nothing else
- Every user's most recent order is `train` or `test`, never `prior`, so the split is intact

No cleaning step needed going into EDA — I was expecting to find at least a few missing or null rows
given how the sample data behaved, but the full dataset is well-formed.